In [1]:
!pip install pytorch-crf

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from torchcrf import CRF 

In [3]:
class CRFNERHead(nn.Module):
    """
    NER classifier with an optional expandable feedforward network before CRF.
    """
    def __init__(self, hidden_size, label_list, expansion_factor=1):
        """
        Args:
        - hidden_size: Transformer embedding size
        - label_list: List of NER labels
        - expansion_factor: Controls feature expansion (1x, 2x, 0.5x)
        """
        super(CRFNERHead, self).__init__()
        self.label_list = label_list
        self.num_labels = len(label_list)
        self.expanded_size = int(hidden_size * expansion_factor)  # Expand or reduce

        # Expand or reduce features dynamically
        self.ff_layer = nn.Linear(hidden_size, self.expanded_size)
        self.activation = nn.ReLU()

        # Classifier
        self.classifier = nn.Linear(self.expanded_size, self.num_labels)

        # CRF Layer
        self.crf = CRF(num_tags=self.num_labels, batch_first=True)

    def forward(self, sequence_output, labels=None, mask=None):
        """
        Forward pass:
        - sequence_output: Transformer embeddings (batch, seq_len, hidden_size)
        - labels: Ground truth token labels (batch, seq_len)
        - mask: Padding mask (batch, seq_len)

        Returns:
        - Training: CRF loss
        - Inference: Best decoded sequence
        """
        # Expand or reduce feature dimension
        x = self.ff_layer(sequence_output)  # (batch, seq_len, expanded_size)
        x = self.activation(x)  # Apply ReLU

        # Compute token classification logits
        logits = self.classifier(x)  # (batch, seq_len, num_labels)

        if labels is not None:  # Training
            loss = -self.crf(logits, labels, mask=mask, reduction='mean')  # Compute CRF loss
            return loss
        else:  # Inference (decode best tag sequence)
            return self.crf.decode(logits, mask=mask)


In [4]:
class CrossAttentionLayer(nn.Module):
    def __init__(self, hidden_size, person_labels, pii_labels):
        super(CrossAttentionLayer, self).__init__()
        self.person_labels = person_labels  # Indices for PERSONAL NER labels
        self.pii_labels = pii_labels  # Indices for PII-sensitive labels

        self.query_dense = nn.Linear(hidden_size, hidden_size)  # Queries (PERSON)
        self.key_dense = nn.Linear(hidden_size, hidden_size)  # Keys (PII)
        self.value_dense = nn.Linear(hidden_size, hidden_size)  # Values (PII)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, sequence_output, personal_logits, pii_logits):
        batch_size, seq_len, hidden_dim = sequence_output.shape

        # Extract probability scores for Personal & PII tokens
        personal_probs = F.softmax(personal_logits, dim=-1)[..., self.person_labels].sum(dim=-1, keepdim=True)  # (batch, seq_len, 1)
        pii_probs = F.softmax(pii_logits, dim=-1)[..., self.pii_labels].sum(dim=-1, keepdim=True)  # (batch, seq_len, 1)

        # Compute Queries, Keys, and Values
        queries = self.query_dense(sequence_output) * personal_probs  # Queries from Personal Entities
        keys = self.key_dense(sequence_output) * pii_probs  # Keys from PII Entities
        values = self.value_dense(sequence_output) * pii_probs  # Values from PII Entities

        # Compute Attention Scores
        scores = torch.matmul(queries, keys.transpose(-2, -1)) / (hidden_dim ** 0.5)  # (batch, seq_len, seq_len)
        attention_weights = self.softmax(scores)  # Normalize across sequence

        # Compute Final Weighted Sum of Values
        attention_output = torch.matmul(attention_weights, values)  # (batch, seq_len, hidden_size)

        return attention_output, attention_weights


In [5]:
class PrivacyClassificationHead(nn.Module):
    """
    Final classifier with a Feedforward layer before classification.
    """
    def __init__(self, hidden_size, num_classes=2, pooling="max", expansion_factor=2):
        super(PrivacyClassificationHead, self).__init__()
        self.pooling = pooling
        self.expanded_size = hidden_size * expansion_factor  # Expand features

        # Feedforward Layer
        self.ff_layer = nn.Linear(hidden_size, self.expanded_size)
        self.activation = nn.ReLU()  # Non-linearity

        # Classification Layer
        self.fc = nn.Linear(self.expanded_size, num_classes)

    def forward(self, cross_attention_output):
        """
        Inputs:
        - cross_attention_output: (batch, seq_len, hidden_size) - Attention layer output
        
        Returns:
        - logits: (batch, num_classes) - Privacy classification logits
        """
        if self.pooling == "mean":
            pooled_output = torch.mean(cross_attention_output, dim=1)
        elif self.pooling == "max":
            pooled_output, _ = torch.max(cross_attention_output, dim=1)
        elif self.pooling == "cls":
            pooled_output = cross_attention_output[:, 0, :]
        else:
            raise ValueError("Invalid pooling type. Choose from ['max', 'mean', 'cls']")

        # Apply Feedforward Layer
        x = self.ff_layer(pooled_output)  # (batch, expanded_size)
        x = self.activation(x)  # Apply ReLU

        logits = self.fc(x)  # Final classification
        return logits


In [6]:
class PrivacyDetectionModel(nn.Module):
    """
    Full Privacy Detection Model with:
    - ModernBERT as backbone
    - Two independent NER heads (Personal, PII)
    - Cross Attention between NER outputs
    - Privacy classification based on attention output
    """
    def __init__(self, base_model, personal_labels, pii_labels, num_classes=2, expansion_factor=2):
        """
        Args:
        - base_model: Pretrained ModernBERT backbone.
        - personal_labels: List of labels for Personal NER.
        - pii_labels: List of labels for PII-sensitive tokens.
        - num_classes: Number of output classes (default = 2 for binary classification).
        - expansion_factor: Controls feature expansion in FF layer.
        """
        super(PrivacyDetectionModel, self).__init__()
        self.base_model = base_model
        hidden_size = base_model.config.hidden_size

        # Independent NER Heads
        self.personal_ner_head = CRFNERHead(hidden_size, personal_labels)
        self.pii_ner_head = CRFNERHead(hidden_size, pii_labels)

        # Cross-Attention between Personal and PII tokens
        self.cross_attention = CrossAttentionLayer(hidden_size, personal_labels, pii_labels)

        # Final Privacy Classification Head
        self.privacy_classifier = PrivacyClassificationHead(
            hidden_size, num_classes=num_classes, pooling="max", expansion_factor=expansion_factor
        )

    def forward(self, input_ids, attention_mask):
        """
        Forward pass:
        - Extracts ModernBERT features.
        - Runs multiple independent (NER, Cross Attention) layers.
        - Produces final privacy classification.

        Outputs:
        - personal_ner_logits: NER logits for Personal entities.
        - pii_ner_logits: NER logits for PII entities.
        - attention_output: Attention-based relationship representation.
        - privacy_classification_logits: Final document classification logits.
        """
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # (batch, seq_len, hidden_size)

        # Step 1: NER Prediction
        personal_ner_logits = self.personal_ner_head(sequence_output)
        pii_ner_logits = self.pii_ner_head(sequence_output)

        # Step 2: Cross Attention
        attention_output, attention_weights = self.cross_attention(sequence_output, personal_ner_logits, pii_ner_logits)

        # Step 3: Privacy Classification
        privacy_classification_logits = self.privacy_classifier(attention_output)

        return {
            "personal_ner_logits": personal_ner_logits,  
            "pii_ner_logits": pii_ner_logits,  
            "attention_output": attention_output,  
            "attention_weights": attention_weights,  
            "privacy_classification_logits": privacy_classification_logits  
        }


In [8]:
model_name = "answerdotai/ModernBERT-base"
base_model = AutoModel.from_pretrained(model_name)
model = PrivacyDetectionModel(base_model, personal_labels=[2,3,4,5], pii_labels=[7,8,9,10])
print(model)

PrivacyDetectionModel(
  (base_model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (